# Stage 3 — Case Retrieval

*Tujuan*: Mengimplementasikan sistem pengambilan (retrieval system) yang menerima kueri kasus dan mengembalikan kasus historis yang paling mirip dari basis kasus, menggunakan vektorisasi TF-IDF dan Cosine Similarity sebagai baseline, dengan tambahan model machine learning SVM/NB (Support Vector Machine / Naive Bayes) sebagai perbandingan.

---

## Daftar Isi

1. [Data Loading](#1.-Data-Loading)
2. [Train/Test Split](#2.-Train/Test-Split)
3. [TF-IDF Vectorization](#3.-TF-IDF-Vectorization)
4. [Baseline Retrieval — Cosine Similarity](#4.-Baseline-Retrieval-—-Cosine-Similarity)
5. [Retrieval Model — Linear SVM](#5.-Retrieval-Model-—-Linear-SVM)
6. [Retrieval API](#6.-Retrieval-API)
7. [Evaluation Queries](#7.-Evaluation-Queries)
8. [Retrieval Comparison & Analysis](#8.-Retrieval-Comparison-&-Analysis)

**Lampiran (Appendices):**
- [A. Completion Checklist](#A.-Completion-Checklist)

---

# 1. Data Loading

Memuat basis kasus yang telah dipraproses dari Tahap 2.

In [1]:
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Project paths
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EVAL_DIR = PROJECT_ROOT / 'data' / 'eval'
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'

EVAL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load case base
df = pd.read_csv(PROCESSED_DIR / 'cases.csv')
print(f'Case base loaded: {len(df)} cases, {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')

Case base loaded: 44 cases, 11 columns
Columns: ['case_id', 'nomor_putusan', 'pengadilan', 'tanggal_putusan', 'terdakwa', 'pasal', 'ringkasan_fakta', 'amar_putusan', 'pidana_penjara', 'denda', 'text_full']


---

# 2. Train/Test Split

Bagi dataset menjadi set data latih (training set) dan data uji (test set) sebelum melakukan vektorisasi.

*Metodologi*:
 - *Rasio*: 80:20 (sekitar 35 data latih, 9 data uji).
 
 - *Stratifikasi*: Kami menetapkan label pasal UU ITE utama (Pasal 27–37) untuk setiap kasus dan melakukan pembagian secara stratifikasi(stratified split) untuk mempertahankan distribusi label.
 
 - *Tujuan*: Set data latih digunakan untuk mencocokkan (fit) vektoriser TF-IDF dan melatih model SVM. Set data uji berfungsi sebagai kueri evaluasi.

In [2]:
def get_primary_pasal(pasal_str: str) -> str:
    """Extract the primary UU ITE violation article (Pasal 27-37).
    
    Returns the first UU ITE violation article found,
    or 'Lainnya' if none is present.
    """
    if pd.isna(pasal_str):
        return 'Lainnya'
    for part in pasal_str.split(';'):
        m = re.match(r'^\s*(\d+)', part.strip())
        if m:
            num = int(m.group(1))
            if 27 <= num <= 37:
                return f'Pasal {num}'
    return 'Lainnya'


# Assign primary article label to each case
df['primary_pasal'] = df['pasal'].apply(get_primary_pasal)

print('Primary Pasal Distribution:')
print(df['primary_pasal'].value_counts().to_string())
print(f'\nTotal: {len(df)}')

Primary Pasal Distribution:
primary_pasal
Pasal 28    14
Pasal 27    12
Pasal 35    11
Pasal 29     3
Pasal 30     3
Pasal 32     1

Total: 44


In [3]:
# Perform stratified 80:20 split
# For stratification, classes need at least 2 members.
# Merge rare classes (< 2 members) into the largest class for stratification.
strat_labels = df['primary_pasal'].copy()
value_counts = strat_labels.value_counts()
largest_class = value_counts.index[0]  # most common class
rare_classes = value_counts[value_counts < 2].index
strat_labels[strat_labels.isin(rare_classes)] = largest_class

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=strat_labels
)

train_df = train_df.sort_values('case_id').reset_index(drop=True)
test_df = test_df.sort_values('case_id').reset_index(drop=True)

print(f'Train set: {len(train_df)} cases')
print(f'Test set:  {len(test_df)} cases')
print(f'\nTrain distribution:')
print(train_df['primary_pasal'].value_counts().to_string())
print(f'\nTest distribution:')
print(test_df['primary_pasal'].value_counts().to_string())

Train set: 35 cases
Test set:  9 cases

Train distribution:
primary_pasal
Pasal 28    11
Pasal 27    10
Pasal 35     9
Pasal 29     2
Pasal 30     2
Pasal 32     1

Test distribution:
primary_pasal
Pasal 28    3
Pasal 35    2
Pasal 27    2
Pasal 30    1
Pasal 29    1


---

# 3. TF-IDF Vectorization

Cocokkan (fit) vektoriser TF-IDF secara ketat hanya pada set data latih dan transformasikan kedua set data (latih dan uji) tersebut.

*Konfigurasi*:
 - Menggunakan ringkasan_fakta sebagai fitur teks (kolom pengambilan utama / primary retrieval field).

 - Tidak ada penghapusan kata hubung/kata umum (stopword removal) teks sudah dalam bahasa Indonesia dan mengandung istilah-istilah hukum yang spesifik pada domain tersebut.
 
 - Penskalaan TF sublinear (sublinear_tf=True) untuk mengurangi efek dari frekuensi istilah (term frequencies) yang sangat tinggi.

In [4]:
# Fit TF-IDF on training set only
tfidf = TfidfVectorizer(
    sublinear_tf=True,
    max_features=5000,
    ngram_range=(1, 2),  # unigrams + bigrams
)

# Fit on training ringkasan_fakta
train_texts = train_df['ringkasan_fakta'].fillna('').tolist()
test_texts = test_df['ringkasan_fakta'].fillna('').tolist()

X_train = tfidf.fit_transform(train_texts)
X_test = tfidf.transform(test_texts)

print(f'TF-IDF vocabulary size: {len(tfidf.vocabulary_)}')
print(f'Train matrix shape: {X_train.shape}')
print(f'Test matrix shape:  {X_test.shape}')

# Show top features
feature_names = tfidf.get_feature_names_out()
print(f'\nSample features (first 20): {list(feature_names[:20])}')

TF-IDF vocabulary size: 3012
Train matrix shape: (35, 3012)
Test matrix shape:  (9, 3012)

Sample features (first 20): ['00', '00 000', '00 wi', '00 wib', '00 wita', '000', '000 000', '000 kepada', '000 lima', '000 satu', '000 sepuluh', '000 seratus', '000 sueratus', '02', '02 juli', '04', '04 2021', '04 desember', '043801020879501', '043801020879501 atas']


---

# 4. Baseline Retrieval — Cosine Similarity

Logika utama pengambilan (retrieval): menghitung Cosine Similarity (kemiripan kosinus) antara vektor kueri dan vektor basis kasus latih, kemudian mengurutkannya berdasarkan skor kemiripan.

Konfigurasi Top-K: k=5 — sebuah metrik standar untuk evaluasi pengambilan, yang memberikan pandangan seimbang terhadap kasus-kasus paling relevan tanpa membebani pengguna dengan informasi yang berlebihan atau melemahkan ambang batas relevansi.

In [5]:
def retrieve_cosine(query_vector, k=5):
    """Retrieve top-K similar cases using Cosine Similarity.
    
    Args:
        query_vector: TF-IDF vector of the query (sparse matrix, 1 x n_features).
        k: Number of top results to return.
    
    Returns:
        DataFrame with case_id, similarity_score, pasal, ringkasan_fakta.
    """
    # Compute cosine similarity against training set
    sim_scores = cosine_similarity(query_vector, X_train).flatten()
    
    # Rank by similarity (descending)
    top_indices = sim_scores.argsort()[::-1][:k]
    
    results = []
    for idx in top_indices:
        results.append({
            'case_id': train_df.iloc[idx]['case_id'],
            'similarity_score': round(float(sim_scores[idx]), 4),
            'pasal': train_df.iloc[idx]['pasal'],
            'primary_pasal': train_df.iloc[idx]['primary_pasal'],
            'ringkasan_fakta': train_df.iloc[idx]['ringkasan_fakta'],
        })
    return pd.DataFrame(results)


# Quick test: use the first test case as a query
query_vec = X_test[0]
query_case = test_df.iloc[0]
print(f'Query: {query_case["case_id"]} (primary_pasal: {query_case["primary_pasal"]})')
print(f'Query text: {query_case["ringkasan_fakta"][:150]}...\n')

results = retrieve_cosine(query_vec, k=5)
print('Top-5 results:')
for _, row in results.iterrows():
    print(f'  {row["case_id"]} | sim={row["similarity_score"]:.4f} | {row["primary_pasal"]} | {str(row["ringkasan_fakta"])[:80]}...')

Query: case_001 (primary_pasal: Pasal 35)
Query text: bahwa ia terdakwa toppan bersama dengan saksi muharram syukri (penuntutan di lakukan terpisah) dan saksi sarwan (penuntutan di lakukan terpisah) pada ...

Top-5 results:
  case_029 | sim=0.2558 | Pasal 30 | bahwa para terdakwa diajukan ke muka perksidangan oleh penuntut umum dengan sura...
  case_023 | sim=0.2121 | Pasal 28 | bahwa terdakwa jocky chen pada hari minggu tangngal 12 mei 2019 sekitar jam 11:0...
  case_037 | sim=0.2020 | Pasal 28 | bahwa terdakwa meswono pada hari- hari yang sudah tidak dapat diingat kembali de...
  case_036 | sim=0.2007 | Pasal 35 | bahwa teguh bin sidi, pada hari selasa tanggal 9 juni 2020 sekitar jam 09.38 wib...
  case_013 | sim=0.1803 | Pasal 27 | bahwa terdakwa diajukan ke persidangan adalah karena gdidakwa : kesatu ---------...


---

# 5. Retrieval Model — Linear SVM

Latih pengklasifikasi Linear SVM menggunakan fitur TF-IDF untuk memprediksi pasal pelanggaran UU ITE utama. Model ini disertakan untuk memenuhi persyaratan pengambilan/klasifikasi (retrieval/classification) machine learning dari tugas tersebut.

SVM dapat digunakan untuk pengambilan dengan cara:
 1. Memprediksi kategori pasal utama dari sebuah kueri.

 2. Mengembalikan kasus-kasus latih yang memiliki kategori prediksi yang sama, diurutkan berdasarkan skor fungsi keputusan (decision function scores) SVM.

In [6]:
# Encode labels for SVM training
le = LabelEncoder()
y_train = le.fit_transform(train_df['primary_pasal'])
y_test = le.transform(test_df['primary_pasal'])

print(f'Label classes: {list(le.classes_)}')
print(f'Train labels shape: {y_train.shape}')
print(f'Test labels shape:  {y_test.shape}')

Label classes: ['Pasal 27', 'Pasal 28', 'Pasal 29', 'Pasal 30', 'Pasal 32', 'Pasal 35']
Train labels shape: (35,)
Test labels shape:  (9,)


In [7]:
# Train Linear SVM
svm = LinearSVC(max_iter=10000, random_state=42)
svm.fit(X_train, y_train)

# Evaluate on test set
train_acc = svm.score(X_train, y_train)
test_acc = svm.score(X_test, y_test)

print(f'SVM Training accuracy: {train_acc:.2%}')
print(f'SVM Test accuracy:     {test_acc:.2%}')

# Show predictions vs actual for test cases
y_pred = svm.predict(X_test)
print(f'\nTest predictions vs actual:')
for i in range(len(test_df)):
    actual = le.inverse_transform([y_test[i]])[0]
    predicted = le.inverse_transform([y_pred[i]])[0]
    match = '✅' if actual == predicted else '❌'
    print(f'  {test_df.iloc[i]["case_id"]}: actual={actual}, predicted={predicted} {match}')

SVM Training accuracy: 100.00%
SVM Test accuracy:     33.33%

Test predictions vs actual:
  case_001: actual=Pasal 35, predicted=Pasal 28 ❌
  case_006: actual=Pasal 28, predicted=Pasal 35 ❌
  case_014: actual=Pasal 27, predicted=Pasal 28 ❌
  case_019: actual=Pasal 27, predicted=Pasal 35 ❌
  case_022: actual=Pasal 28, predicted=Pasal 28 ✅
  case_025: actual=Pasal 30, predicted=Pasal 27 ❌
  case_032: actual=Pasal 29, predicted=Pasal 27 ❌
  case_033: actual=Pasal 35, predicted=Pasal 35 ✅
  case_043: actual=Pasal 28, predicted=Pasal 28 ✅


In [8]:
def retrieve_svm(query_vector, k=5):
    """Retrieve top-K cases using SVM-based classification.
    
    Strategy:
    1. Predict the primary pasal category of the query.
    2. Return training cases matching the predicted category,
       ranked by SVM decision function score (confidence).
    3. If fewer than k cases match, fill with cosine-similarity fallback.
    
    Args:
        query_vector: TF-IDF vector of the query.
        k: Number of top results to return.
    
    Returns:
        DataFrame with case_id, similarity_score, pasal, ringkasan_fakta, predicted_label.
    """
    # Predict category
    pred_label_idx = svm.predict(query_vector)[0]
    pred_label = le.inverse_transform([pred_label_idx])[0]
    
    # Get decision function scores for ranking
    decision_scores = svm.decision_function(X_train)
    if len(le.classes_) == 2:
        # Binary case: single score per sample
        scores = decision_scores
    else:
        # Multi-class: get scores for the predicted class
        class_idx = list(le.classes_).index(pred_label)
        scores = decision_scores[:, class_idx]
    
    # Filter to matching category and rank by score
    matching_mask = train_df['primary_pasal'] == pred_label
    matching_indices = np.where(matching_mask)[0]
    matching_scores = scores[matching_indices]
    
    # Sort by score descending
    sorted_order = matching_scores.argsort()[::-1][:k]
    top_indices = matching_indices[sorted_order]
    
    results = []
    for idx in top_indices:
        # Normalize score to [0, 1] range for comparison
        raw_score = float(scores[idx])
        norm_score = round(1 / (1 + np.exp(-raw_score)), 4)  # sigmoid normalization
        results.append({
            'case_id': train_df.iloc[idx]['case_id'],
            'similarity_score': norm_score,
            'pasal': train_df.iloc[idx]['pasal'],
            'primary_pasal': train_df.iloc[idx]['primary_pasal'],
            'ringkasan_fakta': train_df.iloc[idx]['ringkasan_fakta'],
            'predicted_label': pred_label,
        })
    
    # Fill with cosine fallback if needed
    if len(results) < k:
        cosine_results = retrieve_cosine(query_vector, k=k)
        existing_ids = {r['case_id'] for r in results}
        for _, row in cosine_results.iterrows():
            if row['case_id'] not in existing_ids and len(results) < k:
                entry = row.to_dict()
                entry['predicted_label'] = pred_label
                results.append(entry)
    
    return pd.DataFrame(results)


# Quick test
print(f'Query: {query_case["case_id"]} (actual: {query_case["primary_pasal"]})')
svm_results = retrieve_svm(query_vec, k=5)
print(f'\nSVM Top-5 results:')
for _, row in svm_results.iterrows():
    print(f'  {row["case_id"]} | score={row["similarity_score"]:.4f} | {row["primary_pasal"]} | predicted={row["predicted_label"]}')

Query: case_001 (actual: Pasal 35)

SVM Top-5 results:
  case_011 | score=0.6373 | Pasal 28 | predicted=Pasal 28
  case_037 | score=0.6340 | Pasal 28 | predicted=Pasal 28
  case_038 | score=0.6305 | Pasal 28 | predicted=Pasal 28
  case_003 | score=0.6301 | Pasal 28 | predicted=Pasal 28
  case_004 | score=0.6289 | Pasal 28 | predicted=Pasal 28


---

# 6. Retrieval API

Sebuah fungsi retrieve() yang terpadu dan dapat digunakan kembali yang merangkum seluruh tahapan (pipeline):
 1. Mempraproses teks kueri.
 2. Melakukan vektorisasi menggunakan model TF-IDF yang telah dicocokkan (fitted).
 3. Melakukan pengambilan menggunakan metode yang dipilih (cosine atau svm).
 4. Mengembalikan hasil top-K berperingkat beserta kolom-kolom (fields) utamanya.

In [9]:
def retrieve(query_text: str, k: int = 5, method: str = 'cosine') -> pd.DataFrame:
    """Formal Retrieval API.
    
    Accepts raw query text and returns the top-K most similar cases
    from the training case base.
    
    Args:
        query_text: Raw factual description of the query case.
        k: Number of top results to return (default: 5).
        method: Retrieval method — 'cosine' (baseline) or 'svm'.
    
    Returns:
        DataFrame with columns:
        - case_id: Identifier of the retrieved case.
        - similarity_score: Relevance score (0.0 to 1.0, descending).
        - pasal: Charged articles of the retrieved case.
        - ringkasan_fakta: Factual summary of the retrieved case.
    """
    # Step 1: Vectorize the query
    query_vector = tfidf.transform([query_text])
    
    # Step 2: Retrieve
    if method == 'cosine':
        return retrieve_cosine(query_vector, k=k)
    elif method == 'svm':
        return retrieve_svm(query_vector, k=k)
    else:
        raise ValueError(f'Unknown method: {method}. Use "cosine" or "svm".')


# Quick test with raw text
sample_query = test_df.iloc[0]['ringkasan_fakta']
result = retrieve(sample_query, k=3, method='cosine')
print('retrieve() API test (cosine, k=3):')
print(result[['case_id', 'similarity_score', 'primary_pasal']].to_string(index=False))

print()
result_svm = retrieve(sample_query, k=3, method='svm')
print('retrieve() API test (svm, k=3):')
print(result_svm[['case_id', 'similarity_score', 'primary_pasal']].to_string(index=False))

retrieve() API test (cosine, k=3):
 case_id  similarity_score primary_pasal
case_029            0.2558      Pasal 30
case_023            0.2121      Pasal 28
case_037            0.2020      Pasal 28

retrieve() API test (svm, k=3):
 case_id  similarity_score primary_pasal
case_011            0.6373      Pasal 28
case_037            0.6340      Pasal 28
case_038            0.6305      Pasal 28


---

# 7. Evaluation Queries

Jalankan API pengambilan (retrieval API) terhadap seluruh kasus uji dan buat/hasilkan file data/eval/queries.json dengan label kebenaran dasar (ground-truth labels) yang eksplisit.

Relevansi kebenaran dasar (Ground-truth relevance) didefinisikan menggunakan kecocokan kategori pasal utama: sebuah kasus yang diambil dianggap relevan jika primary_pasal (pasal utamanya) cocok dengan primary_pasal dari kasus kueri.

In [10]:
# Build evaluation queries from test set
eval_queries = []

for i in range(len(test_df)):
    row = test_df.iloc[i]
    
    # Ground truth: training cases with the same primary_pasal
    gt_cases = train_df[train_df['primary_pasal'] == row['primary_pasal']]['case_id'].tolist()
    
    eval_queries.append({
        'query_id': row['case_id'],
        'query_text': row['ringkasan_fakta'],
        'expected_pasal': row['primary_pasal'],
        'expected_similar_cases': gt_cases,
    })

# Save queries.json
queries_path = EVAL_DIR / 'queries.json'
with open(queries_path, 'w', encoding='utf-8') as f:
    json.dump(eval_queries, f, ensure_ascii=False, indent=2)

print(f'Saved {len(eval_queries)} evaluation queries to {queries_path}')
for q in eval_queries:
    print(f'  {q["query_id"]}: expected={q["expected_pasal"]}, ground_truth_count={len(q["expected_similar_cases"])}')

Saved 9 evaluation queries to D:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\data\eval\queries.json
  case_001: expected=Pasal 35, ground_truth_count=9
  case_006: expected=Pasal 28, ground_truth_count=11
  case_014: expected=Pasal 27, ground_truth_count=10
  case_019: expected=Pasal 27, ground_truth_count=10
  case_022: expected=Pasal 28, ground_truth_count=11
  case_025: expected=Pasal 30, ground_truth_count=2
  case_032: expected=Pasal 29, ground_truth_count=2
  case_033: expected=Pasal 35, ground_truth_count=9
  case_043: expected=Pasal 28, ground_truth_count=11


In [11]:
# Run retrieval for all evaluation queries
all_results = {}

for q in eval_queries:
    query_text = q['query_text']
    query_id = q['query_id']
    
    # Cosine retrieval
    cosine_res = retrieve(query_text, k=5, method='cosine')
    
    # SVM retrieval
    svm_res = retrieve(query_text, k=5, method='svm')
    
    all_results[query_id] = {
        'query_id': query_id,
        'expected_pasal': q['expected_pasal'],
        'expected_similar_cases': q['expected_similar_cases'],
        'cosine_results': cosine_res[['case_id', 'similarity_score', 'primary_pasal']].to_dict('records'),
        'svm_results': svm_res[['case_id', 'similarity_score', 'primary_pasal']].to_dict('records'),
    }

# Save retrieval outputs
output_path = RESULTS_DIR / 'retrieval_baseline.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f'Saved retrieval results to {output_path}')
print(f'Queries processed: {len(all_results)}')

Saved retrieval results to D:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\data\results\retrieval_baseline.json
Queries processed: 9


---

# 8. Retrieval Comparison & Analysis

Bandingkan baseline TF-IDF + Cosine Similarity dengan model TF-IDF + Linear SVM.

In [12]:
# Compute precision@5 for each method
# A retrieved case is 'relevant' if its primary_pasal matches the query's.

cosine_precision_scores = []
svm_precision_scores = []

print('Per-Query Results (Precision@5):')
print('=' * 80)

for q in eval_queries:
    qid = q['query_id']
    expected = q['expected_pasal']
    gt_set = set(q['expected_similar_cases'])
    
    res = all_results[qid]
    
    # Cosine precision@5
    cos_hits = sum(1 for r in res['cosine_results'] if r['case_id'] in gt_set)
    cos_p5 = cos_hits / 5
    cosine_precision_scores.append(cos_p5)
    
    # SVM precision@5
    svm_hits = sum(1 for r in res['svm_results'] if r['case_id'] in gt_set)
    svm_p5 = svm_hits / 5
    svm_precision_scores.append(svm_p5)
    
    print(f'  {qid} (expected: {expected}):')
    print(f'    Cosine: P@5={cos_p5:.2f} — retrieved: {[r["case_id"] + " (" + r["primary_pasal"] + ")" for r in res["cosine_results"]]}')
    print(f'    SVM:    P@5={svm_p5:.2f} — retrieved: {[r["case_id"] + " (" + r["primary_pasal"] + ")" for r in res["svm_results"]]}')
    print()

avg_cos_p5 = np.mean(cosine_precision_scores)
avg_svm_p5 = np.mean(svm_precision_scores)

print('=' * 80)
print(f'Average Precision@5 — Cosine: {avg_cos_p5:.2f}')
print(f'Average Precision@5 — SVM:    {avg_svm_p5:.2f}')

Per-Query Results (Precision@5):
  case_001 (expected: Pasal 35):
    Cosine: P@5=0.20 — retrieved: ['case_029 (Pasal 30)', 'case_023 (Pasal 28)', 'case_037 (Pasal 28)', 'case_036 (Pasal 35)', 'case_013 (Pasal 27)']
    SVM:    P@5=0.00 — retrieved: ['case_011 (Pasal 28)', 'case_037 (Pasal 28)', 'case_038 (Pasal 28)', 'case_003 (Pasal 28)', 'case_004 (Pasal 28)']

  case_006 (expected: Pasal 28):
    Cosine: P@5=0.40 — retrieved: ['case_029 (Pasal 30)', 'case_037 (Pasal 28)', 'case_024 (Pasal 35)', 'case_036 (Pasal 35)', 'case_023 (Pasal 28)']
    SVM:    P@5=0.00 — retrieved: ['case_015 (Pasal 35)', 'case_024 (Pasal 35)', 'case_036 (Pasal 35)', 'case_021 (Pasal 35)', 'case_028 (Pasal 35)']

  case_014 (expected: Pasal 27):
    Cosine: P@5=0.40 — retrieved: ['case_009 (Pasal 35)', 'case_030 (Pasal 27)', 'case_034 (Pasal 35)', 'case_035 (Pasal 30)', 'case_018 (Pasal 27)']
    SVM:    P@5=0.00 — retrieved: ['case_011 (Pasal 28)', 'case_037 (Pasal 28)', 'case_038 (Pasal 28)', 'case_003 (P

In [13]:
# Comparison table
comparison = pd.DataFrame([
    {
        'Retrieval Method': 'TF-IDF + Cosine Similarity',
        'Avg Precision@5': f'{avg_cos_p5:.2f}',
        'Strengths': 'Unsupervised; no training labels needed; captures text similarity directly',
        'Weaknesses': 'Sensitive to vocabulary overlap; cannot leverage category structure',
    },
    {
        'Retrieval Method': 'TF-IDF + Linear SVM',
        'Avg Precision@5': f'{avg_svm_p5:.2f}',
        'Strengths': 'Leverages labeled categories; generalizes across similar pasal groups',
        'Weaknesses': 'Requires labeled training data; less flexible for unseen categories',
    },
])

print('\nRetrieval Method Comparison')
print('=' * 80)
print(comparison.to_string(index=False))


Retrieval Method Comparison
          Retrieval Method Avg Precision@5                                                                  Strengths                                                          Weaknesses
TF-IDF + Cosine Similarity            0.24 Unsupervised; no training labels needed; captures text similarity directly Sensitive to vocabulary overlap; cannot leverage category structure
       TF-IDF + Linear SVM            0.33      Leverages labeled categories; generalizes across similar pasal groups Requires labeled training data; less flexible for unseen categories


In [14]:
# Final summary
print('=' * 60)
print('STAGE 3 — CASE RETRIEVAL SUMMARY')
print('=' * 60)
print(f'\n  Train set size:       {len(train_df)}')
print(f'  Test set size:        {len(test_df)}')
print(f'  TF-IDF vocabulary:    {len(tfidf.vocabulary_)}')
print(f'  Evaluation queries:   {len(eval_queries)}')
print(f'  Top-K:                5')
print(f'\n  Avg Precision@5 (Cosine): {avg_cos_p5:.2f}')
print(f'  Avg Precision@5 (SVM):    {avg_svm_p5:.2f}')
print(f'  SVM Test Accuracy:        {test_acc:.2%}')
print(f'\n  Outputs:')
print(f'    queries.json:           {queries_path}')
print(f'    retrieval_baseline.json: {output_path}')
print(f'\n' + '=' * 60)

STAGE 3 — CASE RETRIEVAL SUMMARY

  Train set size:       35
  Test set size:        9
  TF-IDF vocabulary:    3012
  Evaluation queries:   9
  Top-K:                5

  Avg Precision@5 (Cosine): 0.24
  Avg Precision@5 (SVM):    0.33
  SVM Test Accuracy:        33.33%

  Outputs:
    queries.json:           D:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\data\eval\queries.json
    retrieval_baseline.json: D:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\data\results\retrieval_baseline.json



---

# Appendices

## A. Completion Checklist

 - [x] Representasi TF-IDF telah diimplementasikan.
 - [x] Pembagian data latih-uji (train-test split) telah diimplementasikan (80:20).
 - [x] Model pengambilan Linear SVM telah diimplementasikan.
 - [x] Fungsi retrieve() telah diimplementasikan dan diuji.
 - [x] 5–10 kueri evaluasi telah disiapkan.
 - [x] Label kebenaran dasar (ground-truth labels) telah disiapkan.
 - [x] Keluaran/hasil pengambilan telah berhasil dibuat dan disimpan.
 - [x] queries.json berisi label kebenaran dasar yang eksplisit.

---

*Stage 3 (Case Retrieval) is considered complete when all checklist items above are verified.*